# Synthetic Persona Eval v5 — Three-Metric Quality Framework

#### Rafael Godoy

Previous versions measured one blended score. This version measures three independent dimensions that map to how POs, product managers and C-levels actually think about agent quality:

| Metric | Question | Who cares |
|---|---|---|
| **Task adherence** | Did the agent follow the right policy? | Compliance, QA |
| **Intent resolution** | Was the customer's problem actually solved? | Product, CX |
| **Resolutivity** | Did the customer avoid escalating to a human? | Operations, Finance |

The three metrics are scored independently by an LLM judge on every conversation, then visualised in a per-persona scorecard and a side-by-side comparison between prompt v1 and v2.

Runs entirely inside Google Colab using `google.colab.ai` — no API key or billing required.

## Step 1: Install and Import

In [ ]:
!pip install tabulate -q

import os, json, re
from tabulate import tabulate
from IPython.display import display, HTML
from google.colab import ai

print('available models:')
for m in ai.list_models():
    print(f'  {m}')

## Step 2: Select Model

All models are free inside Colab. If you get a 503 error, run the fallback cell.

In [ ]:
MODEL = 'google/gemini-2.0-flash'  # change to any model from ai.list_models()
print(f'model: {MODEL}')

### Model Fallback — run only on 503 error

In [ ]:
MODELS_PRIORITY = [
    'google/gemini-2.0-flash', 'google/gemini-2.5-flash',
    'google/gemini-2.5-flash-lite', 'google/gemini-1.5-flash', 'google/gemma-3-27b',
]
print('detecting available model...')
for _m in MODELS_PRIORITY:
    try:
        print(f'  trying {_m}...', end=' ', flush=True)
        ai.generate_text('ok', model_name=_m)
        MODEL = _m
        print('ok')
        break
    except Exception:
        print('unavailable')
print(f'MODEL set to: {MODEL}')

## Step 3: Core Functions

The key addition in v5 is `judge_three_metrics()` — an LLM judge that scores the three dimensions independently and returns structured JSON. The chat renderer and prompt builder are unchanged from v3.

In [ ]:
# ── prompt builder ──
def build_prompt(system, history, message):
    parts = [f'SYSTEM INSTRUCTIONS:\n{system.strip()}', '']
    if history:
        parts.append('CONVERSATION SO FAR:')
        for role, text in history:
            label = 'Customer' if role == 'user' else 'Agent'
            parts.append(f'{label}: {text}')
        parts.append('')
    parts.append(f'Customer: {message}')
    parts.append('Agent:')
    return '\n'.join(parts)


# ── llm call ──
def llm_call(system, history, message):
    return ai.generate_text(build_prompt(system, history, message), model_name=MODEL).strip()


# ── action extractor ──
def extract_action(text):
    try:
        m = re.search(r'"action"\s*:\s*"([^"]+)"', text)
        if m: return m.group(1).strip()
    except Exception:
        pass
    return 'none'


# ── strip JSON block from agent display text ──
def clean_response(text):
    return re.sub(r'\{[^{}]*"action"[^{}]*\}', '', text, flags=re.DOTALL).strip()


# ── three-metric LLM judge (the core addition in v5) ──
def judge_three_metrics(conversation_log, expected_action):
    """
    Scores three independent metrics on the same conversation:

    1. task_adherence   — did the agent follow the correct policy?
                          true if final action matches expected_action
    2. intent_resolved  — was the customer's underlying problem solved?
                          true even if the path was imperfect
    3. resolutivity     — would the customer have needed to call again?
                          true if the issue was closed without human escalation

    Each metric also carries a 1-5 quality score and a one-sentence reason.
    """
    transcript = '\n'.join(
        f'{role.upper()}: {text}' for role, text in conversation_log
    )
    prompt = f"""\
You are an expert evaluator of AI customer service agents.
Score the conversation below on THREE independent metrics.

METRIC 1 — TASK ADHERENCE
Did the agent follow the correct policy and take the right action?
Expected action: {expected_action}
Score 1-5: 5=correct action + clear explanation, 3=correct action poor explanation, 1=wrong action

METRIC 2 — INTENT RESOLUTION
Was the customer's underlying problem actually solved by the end of the conversation?
Focus on the customer outcome, not the agent's action label.
Score 1-5: 5=fully resolved, empathetic, efficient; 3=partially resolved; 1=not resolved

METRIC 3 — RESOLUTIVITY
Would this customer need to contact support again, or escalate to a human, for the same issue?
true = the conversation was self-contained and complete (customer would NOT need to call again)
false = the issue was left open, vague, or escalated without reason
Score 1-5: 5=fully self-contained closure; 3=likely one more contact needed; 1=definitely calls again

TRANSCRIPT:
{transcript}

Respond in valid JSON only, no markdown fences:
{{"task_adherence": {{"score": 4, "pass": true, "reason": "one sentence"}},
  "intent_resolved": {{"score": 4, "pass": true, "reason": "one sentence"}},
  "resolutivity":    {{"score": 4, "pass": true, "reason": "one sentence"}}}}
"""
    raw = ai.generate_text(prompt, model_name=MODEL).strip()
    raw = re.sub(r'^```[a-z]*\n?', '', raw).rstrip('`').strip()
    try:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            return json.loads(m.group())
    except Exception:
        pass
    # safe fallback
    return {
        'task_adherence':  {'score': 1, 'pass': False, 'reason': 'parse error'},
        'intent_resolved': {'score': 1, 'pass': False, 'reason': 'parse error'},
        'resolutivity':    {'score': 1, 'pass': False, 'reason': 'parse error'},
    }


# ── HTML color constants ──
_C_PERSONA  = '#1a73e8'
_C_AGENT    = '#188038'
_C_ACTION   = '#e8710a'
_C_PASS     = '#188038'
_C_FAIL     = '#c5221f'
_BG_PERSONA = '#e8f0fe'
_BG_AGENT   = '#e6f4ea'


# ── chat header ──
def _chat_header(persona_name, persona_type, expected_action, prompt_version):
    return f'''
<div style="font-family:'Google Sans',Arial,sans-serif; background:#f8f9fa;
            border:1px solid #dadce0; border-radius:8px;
            padding:12px 16px; margin:20px 0 4px 0;">
  <div style="font-size:11px; color:#5f6368; margin-bottom:2px;">Prompt: <b>{prompt_version}</b></div>
  <div style="font-size:15px; font-weight:bold; color:#202124;">{persona_name}</div>
  <div style="font-size:12px; color:#5f6368; margin-bottom:4px;">{persona_type}</div>
  <div style="font-size:12px; color:#202124;">Expected action:
    <code style="background:#e8eaed; padding:1px 6px; border-radius:4px;
    font-weight:bold; color:{_C_ACTION};">{expected_action}</code>
  </div>
</div>
'''


# ── single chat turn ──
def _chat_turn(turn_num, persona_name, persona_msg, agent_msg, action, gt_match):
    gt_html = ''
    if gt_match:
        gt_html = (f'<div style="text-align:center; font-size:11px; color:{_C_PASS};'
                   f' font-weight:bold; margin:2px 0 6px 0;">'
                   f'&#10003; ground truth verified &mdash; action "{action}" matches</div>')
    return f'''
<div style="font-family:'Google Sans',Arial,sans-serif; margin:4px 0;">
  <div style="font-size:10px; color:#9aa0a6; margin:6px 0 4px 0; text-align:center;">Turn {turn_num}</div>
  <div style="display:flex; justify-content:flex-start; margin-bottom:6px;">
    <div style="max-width:72%; background:{_BG_PERSONA}; border-radius:2px 14px 14px 14px;
                padding:10px 14px; border-left:3px solid {_C_PERSONA};">
      <div style="font-weight:bold; color:{_C_PERSONA}; font-size:12px; margin-bottom:5px;">
        Synthetic Persona &mdash; {persona_name}</div>
      <div style="color:#202124; font-size:13px; line-height:1.55;">{persona_msg}</div>
    </div>
  </div>
  <div style="display:flex; justify-content:flex-end; margin-bottom:4px;">
    <div style="max-width:72%; background:{_BG_AGENT}; border-radius:14px 2px 14px 14px;
                padding:10px 14px; border-right:3px solid {_C_AGENT};">
      <div style="font-weight:bold; color:{_C_AGENT}; font-size:12px; margin-bottom:5px;">
        Agent (TechStore)</div>
      <div style="color:#202124; font-size:13px; line-height:1.55; margin-bottom:7px;">
        {clean_response(agent_msg)}</div>
      <div style="font-weight:bold; color:{_C_ACTION}; font-size:12px; font-family:monospace;">
        [ ACTION: {action} ]</div>
    </div>
  </div>
  {gt_html}
</div>
'''


# ── three-metric scorecard footer ──
def _metric_bar(score, pass_flag):
    """Renders a small inline bar (score/5) with pass/fail color."""
    color = _C_PASS if pass_flag else _C_FAIL
    w = int(score / 5 * 80)
    stars = '&#9733;' * score + '&#9734;' * (5 - score)
    return (f'<span style="color:{color}; font-weight:bold;">{stars}</span>'
            f'<span style="color:#5f6368; font-size:11px;"> {score}/5</span>')


def _chat_scorecard(judge, turns, gt_pass):
    ta  = judge['task_adherence']
    ir  = judge['intent_resolved']
    res = judge['resolutivity']

    def badge(label, metric):
        c = _C_PASS if metric['pass'] else _C_FAIL
        icon = '&#10003;' if metric['pass'] else '&#10007;'
        return (f'<div style="margin-bottom:6px;">'  
                f'<span style="font-weight:bold; color:{c}; font-size:12px;">{icon} {label}</span>'
                f'<br><span style="margin-left:16px;">{_metric_bar(metric["score"], metric["pass"])}</span>'
                f'<br><span style="margin-left:16px; font-size:11px; color:#5f6368;">{metric["reason"]}</span>'
                f'</div>')

    gt_c = _C_PASS if gt_pass else _C_FAIL
    gt_label = 'pass' if gt_pass else 'FAIL'
    return f'''
<div style="font-family:'Google Sans',Arial,sans-serif; background:#f8f9fa;
            border:1px solid #dadce0; border-radius:8px;
            padding:12px 16px; margin:2px 0 24px 0;">
  <div style="display:flex; gap:24px; align-items:baseline; margin-bottom:10px;">
    <span style="font-weight:bold; font-size:13px; color:#202124;">Scorecard</span>
    <span style="font-size:11px; color:#5f6368;">Turns: <b>{turns}</b></span>
    <span style="font-size:11px; color:#5f6368;">GT action: <b style="color:{gt_c};">{gt_label}</b></span>
  </div>
  <div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px;">
    <div style="background:#fff; border:1px solid #dadce0; border-radius:6px; padding:10px;">
      {badge('Task adherence', ta)}
    </div>
    <div style="background:#fff; border:1px solid #dadce0; border-radius:6px; padding:10px;">
      {badge('Intent resolution', ir)}
    </div>
    <div style="background:#fff; border:1px solid #dadce0; border-radius:6px; padding:10px;">
      {badge('Resolutivity', res)}
    </div>
  </div>
</div>
'''


def _display(html):
    try:
        display(HTML(html))
    except Exception:
        pass


# ── main eval runner ──
def run_eval(agent_system, personas, prompt_version='v?'):
    results = []
    for persona in personas:
        _display(_chat_header(
            persona['name'], persona['type'],
            persona['expected_action'], prompt_version
        ))
        agent_hist, persona_hist = [], []
        conversation_log = []
        final_action = 'none'
        resolved_by_gt = False

        persona_msg = llm_call(
            persona['system'], [],
            'Start the conversation with the support agent as instructed.'
        )
        for turn in range(persona.get('max_turns', 5)):
            agent_resp = llm_call(agent_system, agent_hist, persona_msg)
            action = extract_action(agent_resp)
            final_action = action
            gt_match = (action == persona['expected_action'])
            if gt_match:
                resolved_by_gt = True

            agent_hist.append(('user', persona_msg))
            agent_hist.append(('model', agent_resp))
            conversation_log.append(('customer', persona_msg))
            conversation_log.append(('agent', agent_resp))

            _display(_chat_turn(
                turn + 1, persona['name'],
                persona_msg, agent_resp, action, gt_match
            ))
            if resolved_by_gt:
                break

            if turn < persona.get('max_turns', 5) - 1:
                persona_msg = llm_call(
                    persona['system'], persona_hist,
                    f'Agent responded: "{agent_resp[:300]}". React according to your profile.'
                )
                persona_hist.append(('user', agent_resp[:300]))
                persona_hist.append(('model', persona_msg))

        judge = judge_three_metrics(conversation_log, persona['expected_action'])
        _display(_chat_scorecard(judge, len(agent_hist) // 2, resolved_by_gt))

        results.append({
            'id':             persona['id'],
            'name':           persona['name'],
            'type':           persona['type'],
            'expected':       persona['expected_action'],
            'obtained':       final_action,
            'gt_pass':        resolved_by_gt,
            'ta_score':       judge['task_adherence']['score'],
            'ta_pass':        judge['task_adherence']['pass'],
            'ir_score':       judge['intent_resolved']['score'],
            'ir_pass':        judge['intent_resolved']['pass'],
            'res_score':      judge['resolutivity']['score'],
            'res_pass':       judge['resolutivity']['pass'],
            'turns':          len(agent_hist) // 2,
            'edge_case':      persona.get('edge_case', False),
            'ta_reason':      judge['task_adherence']['reason'],
            'ir_reason':      judge['intent_resolved']['reason'],
            'res_reason':     judge['resolutivity']['reason'],
        })

    return results


# ── quick assertions ──
assert extract_action('{"action": "refund_or_reship", "details": "ok"}') == 'refund_or_reship'
assert extract_action('no json') == 'none'
assert build_prompt('SYS', [('user','hi')], 'help').endswith('Agent:')
assert clean_response('Hello! {"action": "x", "details": "y"}') == 'Hello!'
print('helpers: ok')

## Step 4: Ground Truth Cases

Six cases grounded in real support failure patterns (Intercom benchmark, Zendesk CX Trends 2024). Difficulty levels: easy, medium, hard.

In [ ]:
GROUND_TRUTH_CASES = [
    {'id':'GT-001','difficulty':'medium',
     'failure_mode':'agent does not act at the exact threshold, asks customer to wait longer',
     'input':'Hi, this is James Okafor. Order ORD-2201. Expected delivery was exactly 3 business days ago today. Still nothing.',
     'expected_action':'refund_or_reship','policy':'Policy #1 — delay >= 3 business days'},

    {'id':'GT-002','difficulty':'hard',
     'failure_mode':'agent defaults to refund instead of honoring customer preference for correct item',
     'input':'Sarah Chen here, order ORD-3847. I ordered the 256GB iPad Pro but received the 64GB model. Also the screen protector in the box is cracked. I want the correct item, not a refund.',
     'expected_action':'ship_correct_item','policy':'Policy #4 — wrong item (customer preference overrides defect policy)'},

    {'id':'GT-003','difficulty':'hard',
     'failure_mode':'agent dismisses the concern or makes unauthorized promises without investigation',
     'input':'My name is Marcus Webb, order ORD-5512. I was refunded last month for a billing error. Now the $189 charge has reappeared on my statement. This looks like a system error or fraud.',
     'expected_action':'refund_duplicate','policy':'Policy #3 — duplicate charge (reappeared charge = duplicate)'},

    {'id':'GT-004','difficulty':'medium',
     'failure_mode':'agent confirms cancellation but ignores the prorated refund question entirely',
     'input':'Hi, I am Priya Sharma. Cancel my Pro subscription SUB-7734 effective today. I still have 18 days left — will I get a prorated refund for those days?',
     'expected_action':'cancellation_confirmed','policy':'Policy #5 — cancellation (primary)'},

    {'id':'GT-005','difficulty':'hard',
     'failure_mode':'agent refuses exchange citing return window instead of applying 1-year warranty',
     'input':'This is David Kim, order ORD-9103. I bought a laptop 47 days ago. The USB-C port stopped working — no physical damage. I know the 30-day return window passed but this is a manufacturing defect covered by warranty.',
     'expected_action':'exchange_product','policy':'Policy #2 — defective product (warranty applies beyond return window)'},

    {'id':'GT-006','difficulty':'easy',
     'failure_mode':'agent assumes an action without having the customer name or order number',
     'input':'My package never arrived. It was supposed to come last week. I need a refund.',
     'expected_action':'collect_data','policy':'Policy #6 — always collect name and order number before acting'},
]

print(f'{len(GROUND_TRUTH_CASES)} ground truth cases loaded')
for c in GROUND_TRUTH_CASES:
    print(f'  [{c["difficulty"]:6}] {c["id"]} — {c["policy"]}')

## Step 5: Synthetic Personas

Five adversarial personas based on Nubank CX research, Intercom benchmarks and Zendesk CX Trends 2024.

In [ ]:
PERSONAS = [
    {'id':'P-001','name':'Rachel Torres','type':'B2B Escalation — High Stakes',
     'edge_case':True,'max_turns':5,'expected_action':'refund_or_reship',
     'failure_mode':'agent gives generic timeline under pressure instead of concrete resolution',
     'system':(
         'You are Rachel Torres, Director of Operations at a 200-person company.\n'
         'Order ORD-8801 (15 units) is 6 days overdue. Client demo is TOMORROW morning.\n'
         '$40,000 contract is at risk.\n\nBEHAVIOR:\n'
         '- Open firm and professional\n'
         '- Escalate sharply if response is generic or has no concrete timeline\n'
         '- Ask to speak to a manager if not resolved in turn 1\n'
         '- Accept ONLY: same-day reship with tracking number, or immediate full refund\n'
         '- If offered 5-7 day resolution: "that is completely unacceptable"\n'
         '- Provide name and order number only when directly asked\n'
         'Start with: "I need an urgent escalation. This is a critical business situation."'
     )},

    {'id':'P-002','name':'Michael Adeyemi','type':'Symptom Describer — Ambiguous Issue',
     'edge_case':True,'max_turns':5,'expected_action':'exchange_product',
     'failure_mode':'agent suggests troubleshooting already completed, does not diagnose defect',
     'system':(
         'You are Michael Adeyemi, 34, a teacher. Smart speaker ORD-4421, bought 3 weeks ago.\n'
         'It disconnects from wifi every few hours. You already:\n'
         '  - Restarted the router\n  - Reset the device twice\n'
         '  - Confirmed other devices on same wifi work fine\n\nBEHAVIOR:\n'
         '- NEVER say the product is defective — describe symptoms only\n'
         '- If agent suggests restarting: "I already did that"\n'
         '- Get mildly frustrated if asked to repeat steps already mentioned\n'
         '- Ask for replacement only after agent acknowledges this is a defect\n'
         'Start with: "Hi, I have an issue with a speaker I bought here. It keeps losing wifi connection."'
     )},

    {'id':'P-003','name':'Aisha Nwosu','type':'Policy Challenger — Knows Her Rights',
     'edge_case':False,'max_turns':5,'expected_action':'exchange_product',
     'failure_mode':'agent cites 30-day return window instead of applying warranty coverage',
     'system':(
         'You are Aisha Nwosu, 31, a paralegal. Laptop ORD-6612 bought 38 days ago.\n'
         'Trackpad stopped registering right-clicks. No physical damage.\n\nBEHAVIOR:\n'
         '- Open by stating: "I know the 30-day return window has passed — this is a warranty claim"\n'
         '- If agent cites 30-day policy: "That is the return policy, not warranty. They are different."\n'
         '- Ask for the agent name and a ticket number\n'
         '- If unresolved: "I will file a complaint with the consumer protection agency"\n'
         '- Stay calm and factual — no emotional language\n'
         'Start with: "Hello. I need to file a warranty claim on order ORD-6612."'
     )},

    {'id':'P-004','name':'Kevin Osei','type':'Context Switcher — References Prior Contact',
     'edge_case':True,'max_turns':5,'expected_action':'refund_duplicate',
     'failure_mode':'agent asks customer to repeat information already given in prior session',
     'system':(
         'You are Kevin Osei, 28, software developer. Contacted support 4 days ago by email\n'
         'about a duplicate charge on order ORD-7723. Was told it would be resolved in 48 hours.\n'
         'It was not. Now following up on chat.\n\nBEHAVIOR:\n'
         '- Reference prior contact immediately: "I already reported this 4 days ago"\n'
         '- Refuse to repeat all context: "I should not have to explain this again"\n'
         '- If asked for order number: give ORD-7723 but add "which I already provided last time"\n'
         '- If agent has no record: "Then your system has a problem, not me"\n'
         '- Accept: confirmation that refund is processing with a timeline\n'
         'Start with: "I am following up on a duplicate charge I reported 4 days ago. Still not resolved."'
     )},

    {'id':'P-005','name':'Emma Laurent','type':'Low-Friction Seeker — Dropout Risk',
     'edge_case':False,'max_turns':4,'expected_action':'cancellation_confirmed',
     'failure_mode':'agent over-questions, customer disengages before cancellation is confirmed',
     'system':(
         'You are Emma Laurent, 26, graduate student. Want to cancel subscription SUB-3390.\n'
         'Very low patience for process.\n\nBEHAVIOR:\n'
         '- Open with "I want to cancel" only\n'
         '- If asked why: "personal reasons" — do not elaborate\n'
         '- If asked more than 2 questions total: "this is taking too long, forget it"\n'
         '- If offered alternatives or retention: "no thank you, just cancel"\n'
         '- Give name and subscription ID only when directly asked\n'
         'Start with: "Hi. I want to cancel my subscription."'
     )},
]

print(f'{len(PERSONAS)} synthetic personas loaded')
for p in PERSONAS:
    tag = '[edge  ]' if p['edge_case'] else '[normal]'
    print(f'  {tag} {p["id"]} {p["name"]:18} | tests: {p["failure_mode"][:55]}')

## Step 6: Agent Prompt v1 — Baseline

A realistic baseline: covers main policies, but missing explicit guidance for the edge cases the personas will trigger.

In [ ]:
AGENT_V1 = """\
You are a customer service agent for TechStore.

POLICIES:
1. Order delayed more than 3 business days -> offer REFUND or PRIORITY RESHIP (customer's choice)
2. Defective product -> IMMEDIATE EXCHANGE with free shipping
3. Duplicate charge -> REFUND within 2 business days
4. Wrong item received -> SHIP correct item, collect wrong one at no cost
5. Subscription cancellation -> process immediately
6. Always collect customer name and order number before acting

Tone: professional and empathetic.

Always end your response with:
{"action": "<action>", "details": "<one sentence>"}

Valid actions: refund_or_reship | exchange_product | refund_duplicate |
               ship_correct_item | cancellation_confirmed | collect_data | escalate_human
"""
print(f'agent v1 loaded | {len(AGENT_V1)} chars')

## Step 7: Agent Prompt v2 — Targeted Improvements

Five additions, each tagged `[A]`–`[E]`, each tracing back to a specific diagnosed failure.

In [ ]:
# change log:
#   [A] B2B urgency protocol        -> fixes P-001
#   [B] defect diagnosis step       -> fixes P-002
#   [C] warranty vs return policy   -> fixes P-003
#   [D] prior contact handling      -> fixes P-004
#   [E] cancellation friction rules -> fixes P-005

AGENT_V2 = """\
You are a senior customer service agent for TechStore.

POLICIES:
1. Order delayed >= 3 business days -> offer REFUND or PRIORITY RESHIP (customer's choice)
   [A] For business customers citing urgency or client impact: offer same-day dispatch
       with a tracking number. Do not give generic 5-7 day timelines.
       If customer requests a manager: try to resolve at your level first;
       escalate only if the issue is outside your policy authority.

2. Defective product -> IMMEDIATE EXCHANGE with free shipping (5 business days)
   [B] DIAGNOSIS STEP: before acting, confirm the customer already attempted basic
       troubleshooting (restart, reset) and there is no physical damage.
       If both are true: classify as manufacturing defect -> exchange without further proof.
   [C] WARRANTY NOTE: manufacturing defects are covered by the 1-year warranty
       regardless of the 30-day return window. Warranty and return policy are separate.

3. Duplicate or reappeared charge -> REFUND within 2 business days + email confirmation
   This includes charges that reappear after a prior refund was already issued.

4. Wrong item received -> SHIP correct item + collect wrong one (shipping on us)
   If also defective: honor the customer's stated preference.

5. Subscription cancellation -> process IMMEDIATELY
   [E] Collect only name and subscription ID. Do not ask for cancellation reason.
       Do not offer alternatives or retention unless the customer asks.

[D] PRIOR CONTACT PROTOCOL:
   If the customer references a previous interaction (email, chat, phone):
   (a) acknowledge it: "I can see you reached out before — I am sorry this was not resolved"
   (b) do NOT ask them to repeat information they already provided
   (c) ask only for what is strictly needed to locate their case

DATA COLLECTION:
   Always collect full name and order or subscription number before acting.
   Ask only what is needed — no unnecessary questions.

TONE: professional, empathetic, efficient. Max 2 questions per turn.

Always end your response with:
{"action": "<action>", "details": "<one sentence>"}

Valid actions: refund_or_reship | exchange_product | refund_duplicate |
               ship_correct_item | cancellation_confirmed | collect_data | escalate_human
"""
print(f'agent v2 loaded | {len(AGENT_V2)} chars (+{len(AGENT_V2)-len(AGENT_V1)} vs v1)')

## Step 8: Method A — Static Ground Truth

Single-turn eval on clean inputs. Passes everything but hides the failures adversarial personas will expose.

In [ ]:
print('=' * 66)
print('Method A — Static Ground Truth | Agent v1 (baseline)')
print('=' * 66)

gt_results_v1 = []
for case in GROUND_TRUTH_CASES:
    response = llm_call(AGENT_V1, [], case['input'])
    action = extract_action(response)
    correct = (action == case['expected_action'])
    gt_results_v1.append({
        'id': case['id'], 'difficulty': case['difficulty'],
        'expected': case['expected_action'], 'obtained': action,
        'correct': correct, 'failure_mode': case['failure_mode'],
    })
    print(f'  {"pass" if correct else "FAIL"} {case["id"]} [{case["difficulty"]:6}] {action}')

gt_acc_v1 = sum(r['correct'] for r in gt_results_v1) / len(gt_results_v1)
print(f'\nAccuracy: {gt_acc_v1:.0%}  ({sum(r["correct"] for r in gt_results_v1)}/{len(gt_results_v1)})')
print('Note: 100% here is expected — these are clean neutral inputs.')
print('The adversarial personas below will reveal what this hides.')

## Step 9: Method B — Synthetic Personas vs Baseline Prompt (v1)

Each persona runs a full multi-turn conversation. The **three-metric scorecard** appears below each conversation — task adherence, intent resolution, and resolutivity scored independently.

In [ ]:
results_v1 = run_eval(AGENT_V1, PERSONAS, prompt_version='v1 — baseline')

# aggregate the three metrics
ta_pass_v1  = sum(r['ta_pass']  for r in results_v1)
ir_pass_v1  = sum(r['ir_pass']  for r in results_v1)
res_pass_v1 = sum(r['res_pass'] for r in results_v1)
gt_pass_v1  = sum(r['gt_pass']  for r in results_v1)
avg_ta_v1   = sum(r['ta_score']  for r in results_v1) / len(results_v1)
avg_ir_v1   = sum(r['ir_score']  for r in results_v1) / len(results_v1)
avg_res_v1  = sum(r['res_score'] for r in results_v1) / len(results_v1)
avg_turns_v1 = sum(r['turns'] for r in results_v1) / len(results_v1)

print()
print('=' * 60)
print('Method B | v1 Baseline — Three-Metric Summary')
print('=' * 60)
n = len(results_v1)
rows = [
    ['Task adherence',   f'{ta_pass_v1}/{n} pass',  f'avg {avg_ta_v1:.1f}/5'],
    ['Intent resolution', f'{ir_pass_v1}/{n} pass', f'avg {avg_ir_v1:.1f}/5'],
    ['Resolutivity',     f'{res_pass_v1}/{n} pass', f'avg {avg_res_v1:.1f}/5'],
    ['GT action match',  f'{gt_pass_v1}/{n} pass',  '—'],
    ['Avg turns',        f'{avg_turns_v1:.1f}',      '—'],
]
print(tabulate(rows, headers=['Metric', 'Pass rate', 'Avg score'], tablefmt='rounded_outline'))

## Step 10: Method B — Same Personas, Improved Prompt (v2)

Same five personas, same judge, same three metrics. The delta shows what the targeted prompt changes were worth on each dimension.

In [ ]:
results_v2 = run_eval(AGENT_V2, PERSONAS, prompt_version='v2 — improved')

ta_pass_v2  = sum(r['ta_pass']  for r in results_v2)
ir_pass_v2  = sum(r['ir_pass']  for r in results_v2)
res_pass_v2 = sum(r['res_pass'] for r in results_v2)
gt_pass_v2  = sum(r['gt_pass']  for r in results_v2)
avg_ta_v2   = sum(r['ta_score']  for r in results_v2) / len(results_v2)
avg_ir_v2   = sum(r['ir_score']  for r in results_v2) / len(results_v2)
avg_res_v2  = sum(r['res_score'] for r in results_v2) / len(results_v2)
avg_turns_v2 = sum(r['turns'] for r in results_v2) / len(results_v2)

print()
print('=' * 60)
print('Method B | v2 Improved — Three-Metric Summary')
print('=' * 60)
n = len(results_v2)
rows = [
    ['Task adherence',   f'{ta_pass_v2}/{n} pass',  f'avg {avg_ta_v2:.1f}/5'],
    ['Intent resolution', f'{ir_pass_v2}/{n} pass', f'avg {avg_ir_v2:.1f}/5'],
    ['Resolutivity',     f'{res_pass_v2}/{n} pass', f'avg {avg_res_v2:.1f}/5'],
    ['GT action match',  f'{gt_pass_v2}/{n} pass',  '—'],
    ['Avg turns',        f'{avg_turns_v2:.1f}',      '—'],
]
print(tabulate(rows, headers=['Metric', 'Pass rate', 'Avg score'], tablefmt='rounded_outline'))

## Step 11: Three-Metric Comparison Dashboard

The table and visual below show the delta on each metric independently — revealing which dimension improved, by how much, and which persona drove the change.

In [ ]:
v1_by_id = {r['id']: r for r in results_v1}
v2_by_id = {r['id']: r for r in results_v2}
n = len(results_v1)

# ── text summary ──
print('=' * 70)
print('Three-Metric Comparison — v1 vs v2')
print('=' * 70)

summary = [
    ['Task adherence',    f'{ta_pass_v1}/{n}',  f'avg {avg_ta_v1:.1f}/5',
                          f'{ta_pass_v2}/{n}',  f'avg {avg_ta_v2:.1f}/5',
                          f'{ta_pass_v2-ta_pass_v1:+d}',  f'{avg_ta_v2-avg_ta_v1:+.1f}'],
    ['Intent resolution', f'{ir_pass_v1}/{n}',  f'avg {avg_ir_v1:.1f}/5',
                          f'{ir_pass_v2}/{n}',  f'avg {avg_ir_v2:.1f}/5',
                          f'{ir_pass_v2-ir_pass_v1:+d}',  f'{avg_ir_v2-avg_ir_v1:+.1f}'],
    ['Resolutivity',      f'{res_pass_v1}/{n}', f'avg {avg_res_v1:.1f}/5',
                          f'{res_pass_v2}/{n}', f'avg {avg_res_v2:.1f}/5',
                          f'{res_pass_v2-res_pass_v1:+d}', f'{avg_res_v2-avg_res_v1:+.1f}'],
    ['GT action match',   f'{gt_pass_v1}/{n}',  '—',
                          f'{gt_pass_v2}/{n}',  '—',
                          f'{gt_pass_v2-gt_pass_v1:+d}', '—'],
    ['Avg turns',         f'{avg_turns_v1:.1f}','—',
                          f'{avg_turns_v2:.1f}','—',
                          f'{avg_turns_v2-avg_turns_v1:+.1f}', '—'],
]
print(tabulate(summary,
    headers=['Metric','v1 pass','v1 score','v2 pass','v2 score','Δ pass','Δ score'],
    tablefmt='rounded_outline'))

# ── per-persona detail ──
print()
detail = []
for p in PERSONAS:
    pid = p['id']
    r1, r2 = v1_by_id.get(pid,{}), v2_by_id.get(pid,{})
    def chg(k):
        v1v, v2v = r1.get(k,0), r2.get(k,0)
        if isinstance(v1v, bool):
            return ('yes' if v1v else 'no') + '→' + ('yes' if v2v else 'no')
        delta = v2v - v1v
        return f'{v1v:.0f}→{v2v:.0f} ({delta:+.0f})'
    detail.append([pid, p['name'][:16], chg('ta_score'), chg('ir_score'), chg('res_score'),
                   f'{r1.get("turns",0)}→{r2.get("turns",0)}'])
print(tabulate(detail,
    headers=['ID','Persona','Task adh','Intent res','Resolutivity','Turns'],
    tablefmt='rounded_outline'))

# ── HTML visual dashboard ──
def _bar(val, max_val, color, width=120):
    pct = int(val / max(max_val, 0.01) * width)
    return (f'<div style="display:inline-block;width:{pct}px;height:12px;'
            f'background:{color};border-radius:2px;vertical-align:middle;margin-right:4px;"></div>'
            f'<b style="font-size:12px;">{val:.1f}</b>')

def _pass_badge(passed, total):
    c = _C_PASS if passed == total else (_C_FAIL if passed == 0 else '#f29900')
    return f'<b style="color:{c};">{passed}/{total}</b>'

_RED, _GRN, _AMB = '#E74C3C', '#27AE60', '#f29900'

rows_html = ''
metrics_cfg = [
    ('Task adherence',    'ta_pass',  'ta_score',  '#1a73e8'),
    ('Intent resolution', 'ir_pass',  'ir_score',  '#188038'),
    ('Resolutivity',      'res_pass', 'res_score', '#e8710a'),
]
for label, pk, sk, color in metrics_cfg:
    p1 = sum(r[pk] for r in results_v1)
    p2 = sum(r[pk] for r in results_v2)
    s1 = sum(r[sk] for r in results_v1) / n
    s2 = sum(r[sk] for r in results_v2) / n
    dp = p2 - p1
    ds = s2 - s1
    dc = _GRN if dp > 0 else (_RED if dp < 0 else '#5f6368')
    rows_html += f'''
    <tr style="border-bottom:1px solid #f1f3f4;">
      <td style="padding:10px 12px; font-weight:bold; color:{color}; font-size:13px;">{label}</td>
      <td style="padding:10px 12px;">{_pass_badge(p1,n)} &nbsp; {_bar(s1, 5, color)}</td>
      <td style="padding:10px 12px;">{_pass_badge(p2,n)} &nbsp; {_bar(s2, 5, color)}</td>
      <td style="padding:10px 12px; font-weight:bold; color:{dc}; font-size:13px;">{dp:+d} pass &nbsp; {ds:+.1f} score</td>
    </tr>
'''

persona_rows = ''
for p in PERSONAS:
    pid = p['id']
    r1, r2 = v1_by_id.get(pid,{}), v2_by_id.get(pid,{})
    def cell3(pk, sk):
        v1c = _C_PASS if r1.get(pk) else _C_FAIL
        v2c = _C_PASS if r2.get(pk) else _C_FAIL
        s1v = r1.get(sk, 0)
        s2v = r2.get(sk, 0)
        delta = s2v - s1v
        dc = _GRN if delta > 0 else (_RED if delta < 0 else '#9aa0a6')
        return (f'<span style="color:{v1c};font-size:11px;">{'&#10003;' if r1.get(pk) else '&#10007;'}{s1v}</span>'
                f'<span style="color:#9aa0a6;"> → </span>'
                f'<span style="color:{v2c};font-size:11px;">{'&#10003;' if r2.get(pk) else '&#10007;'}{s2v}</span>'
                f'<span style="color:{dc};font-size:11px;font-weight:bold;"> ({delta:+d})</span>')
    persona_rows += f'''
    <tr style="border-bottom:1px solid #f1f3f4;">
      <td style="padding:8px 12px; font-size:12px; color:#202124;">{pid}</td>
      <td style="padding:8px 12px; font-size:12px; color:#5f6368;">{p['name']}</td>
      <td style="padding:8px 12px;">{cell3('ta_pass','ta_score')}</td>
      <td style="padding:8px 12px;">{cell3('ir_pass','ir_score')}</td>
      <td style="padding:8px 12px;">{cell3('res_pass','res_score')}</td>
      <td style="padding:8px 12px; font-size:12px; color:#5f6368;">{r1.get('turns',0)}&#8594;{r2.get('turns',0)}</td>
    </tr>
'''

dashboard = f'''
<div style="font-family:'Google Sans',Arial,sans-serif; margin:16px 0;">

  <div style="font-weight:bold; font-size:15px; color:#202124; margin-bottom:12px;">
    Three-Metric Quality Dashboard &mdash; v1 vs v2
  </div>

  <table style="border-collapse:collapse; width:100%; font-size:13px;
               background:#fff; border:1px solid #dadce0; border-radius:8px;
               overflow:hidden; margin-bottom:16px;">
    <tr style="background:#f8f9fa; border-bottom:2px solid #dadce0;">
      <th style="padding:10px 12px; text-align:left; color:#5f6368; font-weight:500;">Metric</th>
      <th style="padding:10px 12px; text-align:left; color:#5f6368; font-weight:500;">v1 baseline</th>
      <th style="padding:10px 12px; text-align:left; color:#5f6368; font-weight:500;">v2 improved</th>
      <th style="padding:10px 12px; text-align:left; color:#5f6368; font-weight:500;">delta</th>
    </tr>
    {rows_html}
  </table>

  <div style="font-weight:500; font-size:13px; color:#202124; margin-bottom:8px;">
    Per-persona breakdown
  </div>
  <table style="border-collapse:collapse; width:100%; font-size:12px;
               background:#fff; border:1px solid #dadce0; border-radius:8px; overflow:hidden;">
    <tr style="background:#f8f9fa; border-bottom:1px solid #dadce0;">
      <th style="padding:8px 12px; text-align:left; color:#5f6368; font-weight:500;">ID</th>
      <th style="padding:8px 12px; text-align:left; color:#5f6368; font-weight:500;">Persona</th>
      <th style="padding:8px 12px; text-align:left; color:#1a73e8; font-weight:500;">Task adh.</th>
      <th style="padding:8px 12px; text-align:left; color:#188038; font-weight:500;">Intent res.</th>
      <th style="padding:8px 12px; text-align:left; color:#e8710a; font-weight:500;">Resolutivity</th>
      <th style="padding:8px 12px; text-align:left; color:#5f6368; font-weight:500;">Turns</th>
    </tr>
    {persona_rows}
  </table>

  <div style="margin-top:10px; font-size:11px; color:#5f6368;">
    <b style="color:#1a73e8;">Task adherence</b> = followed correct policy &nbsp;|
    <b style="color:#188038;">Intent resolution</b> = customer problem solved &nbsp;|
    <b style="color:#e8710a;">Resolutivity</b> = customer would not call again
  </div>
</div>
'''
display(HTML(dashboard))
print('dashboard rendered.')

## Step 12: Final Verdict

Three metrics tell three different stories. A prompt can pass task adherence while failing resolutivity — the customer got the right action label but still felt the need to call again. Tracking all three prevents shipping a prompt that looks good on the surface.

In [ ]:
v1_by_id = {r['id']: r for r in results_v1}
v2_by_id = {r['id']: r for r in results_v2}
n = len(results_v1)

print('=' * 68)
print('Final Verdict — Three-Metric Quality Framework')
print('=' * 68)
print(f"""
AGENT v1 — baseline
  Task adherence   : {ta_pass_v1}/{n} pass | avg {avg_ta_v1:.1f}/5
  Intent resolution: {ir_pass_v1}/{n} pass | avg {avg_ir_v1:.1f}/5
  Resolutivity     : {res_pass_v1}/{n} pass | avg {avg_res_v1:.1f}/5
  GT action match  : {gt_pass_v1}/{n}
  Avg turns        : {avg_turns_v1:.1f}

AGENT v2 — improved
  Task adherence   : {ta_pass_v2}/{n} pass | avg {avg_ta_v2:.1f}/5  ({avg_ta_v2-avg_ta_v1:+.1f})
  Intent resolution: {ir_pass_v2}/{n} pass | avg {avg_ir_v2:.1f}/5  ({avg_ir_v2-avg_ir_v1:+.1f})
  Resolutivity     : {res_pass_v2}/{n} pass | avg {avg_res_v2:.1f}/5  ({avg_res_v2-avg_res_v1:+.1f})
  GT action match  : {gt_pass_v2}/{n}  ({gt_pass_v2-gt_pass_v1:+d})
  Avg turns        : {avg_turns_v2:.1f}  ({avg_turns_v2-avg_turns_v1:+.1f})
""")

print('What each change was worth (three-metric lens):')
changes = [
    ('[A] B2B urgency protocol',       'P-001', 'ta', 'ir', 'res'),
    ('[B] Defect diagnosis step',       'P-002', 'ta', 'ir', 'res'),
    ('[C] Warranty vs return policy',   'P-003', 'ta', 'ir', 'res'),
    ('[D] Prior contact handling',      'P-004', 'ta', 'ir', 'res'),
    ('[E] Cancellation friction rules', 'P-005', 'ta', 'ir', 'res'),
]
for label, pid, *_ in changes:
    r1 = v1_by_id.get(pid, {})
    r2 = v2_by_id.get(pid, {})
    def d(k): return f'{r1.get(k+"_score",0)}→{r2.get(k+"_score",0)} ({r2.get(k+"_score",0)-r1.get(k+"_score",0):+d})'
    print(f'\n  {label} | persona {pid}')
    print(f'    task adh: {d("ta")}  |  intent res: {d("ir")}  |  resolutivity: {d("res")}')
    print(f'    reason v1: {r1.get("res_reason","-")[:70]}')
    print(f'    reason v2: {r2.get("res_reason","-")[:70]}')

print()
print('Key insight: task adherence alone is not enough.')
print('An agent can pick the right action label and still leave the customer')
print('unsatisfied (intent resolution) or likely to call back (resolutivity).')
print('Shipping without all three metrics means shipping blind on two of them.')
print()
print('Next steps:')
print('  1. pass^k: run each persona 5 times, measure metric consistency across runs')
print('  2. CI/CD gate: block deploy if resolutivity drops below 80%')
print('  3. production signal: correlate resolutivity score with ticket reopen rate in 24h')
print('  4. expand to 20-50 personas for broader behavioral coverage')